In [4]:
!pip install openpyxl


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\divya\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
data = pd.read_csv("pcos_dataset.csv")
data.head()

,Sl. No,Patient File No.,PCOS (Y/N),Age (yrs),Weight (Kg),Height(Cm),BMI,Blood Group,Pulse rate(bpm),RR (breaths/min),...,Pimples(Y/N),Fast food (Y/N),Reg.Exercise(Y/N),BP _Systolic (mmHg),BP _Diastolic (mmHg),Follicle No. (L),Follicle No. (R),Avg. F size (L) (mm),Avg. F size (R) (mm),Endometrium (mm)
0,193,193,0,30,69.979147,167.708055,23.185569,12,72,22,...,1,0,1,105.483401,76.096379,2,4,10.0,13.0,6.176029
1,360,360,0,36,63.711688,154.055877,25.441392,13,70,18,...,1,0,1,115.883740,79.117243,2,3,13.0,11.0,6.824718
2,10,10,0,36,51.848631,149.059804,23.928264,15,80,20,...,0,0,0,112.219711,80.919417,1,1,14.0,17.0,2.568691
3,278,278,1,29,66.893988,148.628036,27.894935,15,72,18,...,0,0,1,104.619624,69.902681,1,1,12.0,14.0,9.962732
4,71,71,0,33,52.536198,150.767409,23.079564,13,72,18,...,0,0,0,99.175454,70.330461,5,2,11.5,4.7,6.655190


In [6]:
# 🔥 step 1: clean column names
data.columns = data.columns.str.strip()

# 🔥 step 2: drop safely
data.drop([
    "Sl. No",
    "Patient File No.",
    "Blood Group",
    "BMI",
    "Pulse rate(bpm)",
    "RR (breaths/min)",
    "Hb(g/dl)",
    "Marraige Status (Yrs)",
    "Pregnant(Y/N)",
    "No. of abortions",   # fixed
    "I beta-HCG(mIU/mL)",   # fixed spacing
    "II beta-HCG(mIU/mL)",  # fixed spacing
    "FSH(mIU/mL)",
    "LH(mIU/mL)",
    "FSH/LH",
    "Hip(inch)",
    "Waist(inch)",
    "Waist:Hip Ratio",
    "TSH (mIU/L)",
    "AMH(ng/mL)",
    "PRL(ng/mL)",
    "Vit D3 (ng/mL)",
    "PRG(ng/mL)",
    "RBS(mg/dl)",
    "BP _Systolic (mmHg)",
    "BP _Diastolic (mmHg)",
    "Follicle No. (L)",
    "Follicle No. (R)",
    "Avg. F size (L) (mm)",
    "Avg. F size (R) (mm)",
    "Endometrium (mm)",
    "Unnamed: 44"
], axis=1, inplace=True, errors="ignore")  # 🔥 important

print(data.columns)
data.to_csv("new_data.csv", index=False)

Index(['PCOS (Y/N)', 'Age (yrs)', 'Weight (Kg)', 'Height(Cm)', 'Cycle(R/I)',
       'Cycle length(days)', 'I   beta-HCG(mIU/mL)', 'II    beta-HCG(mIU/mL)',
       'Weight gain(Y/N)', 'hair growth(Y/N)', 'Skin darkening (Y/N)',
       'Hair loss(Y/N)', 'Pimples(Y/N)', 'Fast food (Y/N)',
       'Reg.Exercise(Y/N)'],
      dtype='object')


In [7]:
data = data.apply(pd.to_numeric, errors='coerce')
data = data.fillna(data.mean(numeric_only=True))
data["BMI"] = data["Weight (Kg)"] / ((data["Height(Cm)"]/100) ** 2)

In [8]:
X = data[
    [
        'Age (yrs)',
        'Weight (Kg)',
        'Height(Cm)',
        'Fast food (Y/N)',
        'Cycle(R/I)',
        'hair growth(Y/N)',
        'Skin darkening (Y/N)',
        'Hair loss(Y/N)',
        'Pimples(Y/N)',
        'Reg.Exercise(Y/N)',
        'BMI'
    ]
]

y = data['PCOS (Y/N)']

print("✅ Cleaned successfully")
print(X.head())


✅ Cleaned successfully
   Age (yrs)  Weight (Kg)  Height(Cm)  Fast food (Y/N)  Cycle(R/I)  \
0         30    69.979147  167.708055                0           4   
1         36    63.711688  154.055877                0           2   
2         36    51.848631  149.059804                0           4   
3         29    66.893988  148.628036                0           4   
4         33    52.536198  150.767409                0           2   

   hair growth(Y/N)  Skin darkening (Y/N)  Hair loss(Y/N)  Pimples(Y/N)  \
0                 1                     0               0             1   
1                 0                     0               1             1   
2                 0                     0               0             0   
3                 1                     1               1             0   
4                 0                     0               0             0   

   Reg.Exercise(Y/N)        BMI  
0                  1  24.880597  
1                  1  26.844949  
2  

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = LogisticRegression(max_iter = 1000)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [13]:
import os
os.makedirs("models", exist_ok=True)

import pickle
pickle.dump(model, open("models/pcos_model.pkl", "wb"))
pickle.dump(scaler, open("models/scaler.pkl", "wb"))

print("✅ Model and Scaler Saved Successfully")

✅ Model and Scaler Saved Successfully


In [ ]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("AUC-ROC:",roc_auc_score(y_test,y_pred))

Accuracy: 0.8525
F1 Score: 0.7611336032388665
AUC-ROC: 0.8126474424800867
